# Demo 7: Network Impairments on VR/360 Video

**Course**: Future Media Internet
**Duration**: ~10 min
**Environment**: Kaggle Notebook (CPU, NumPy + Matplotlib + SciPy)

## Objective

Show why VR/360 video is MORE sensitive to network degradation than regular HD.
VR requires higher resolution, lower latency, and the equirectangular projection
amplifies artifacts in the viewer's focus area.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
print('Imports OK')


In [ ]:
# Create a synthetic equirectangular VR frame
# VR 360 video is typically 4K+ (3840x1920 or higher)
# Scaled to 960x480 for fast rendering
EH, EW = 480, 960  # equirectangular dimensions

def create_vr_frame():
    """Create a synthetic equirectangular 360 frame with content regions"""
    frame = np.zeros((EH, EW, 3), dtype=np.float32)
    # Horizon line (equator)
    frame[EH//2-2:EH//2+2, :] = 0.8
    # Sky gradient (top half)
    for y in range(EH//2):
        frame[y, :] = (0.3 + 0.5*y/(EH//2), 0.4 + 0.4*y/(EH//2), 0.6 + 0.3*y/(EH//2))
    # Ground (bottom half)
    frame[EH//2:, :] = (0.1, 0.3, 0.1)
    # Objects at different positions (simulating 360 content)
    # Center object (where viewer is likely looking)
    cx, cy = EW//2, EH//2
    for y in range(cy-60, cy+60):
        for x in range(cx-80, cx+80):
            if (x-cx)**2 + (y-cy)**2 < 60**2:
                frame[y, x] = (0.9, 0.2, 0.2)
    # Left object
    for y in range(cy-30, cy+30):
        for x in range(cx-300, cx-200):
            if (x-(cx-250))**2 + (y-cy)**2 < 30**2:
                frame[y, x] = (0.2, 0.9, 0.2)
    # Right object
    for y in range(cy-30, cy+30):
        for x in range(cx+200, cx+300):
            if (x-(cx+250))**2 + (y-cy)**2 < 30**2:
                frame[y, x] = (0.2, 0.2, 0.9)
    # Grid lines for spatial reference
    for x in range(0, EW, 120):
        frame[:, x] *= 0.7
    for y in range(0, EH, 60):
        frame[y, :] *= 0.7
    return frame

vr_frame = create_vr_frame()
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(np.clip(vr_frame, 0, 1))
ax.set_title('VR Equirectangular Frame (960x480)\nRed=Center | Green=Left | Blue=Right', fontsize=13)
ax.axis('off')
plt.show()
print("The equirectangular format stretches poles. Center objects appear", "larger than edge objects when viewed in VR headset.")


In [ ]:
# Network impairment simulators (frame-size-aware)
def packet_loss(frame, rate):
    result = frame.copy()
    H, W = frame.shape[:2]
    bh, bw = 24, 40
    nb_h, nb_w = H//bh, W//bw
    mask = np.random.random((nb_h, nb_w)) < rate
    for i in range(nb_h):
        for j in range(nb_w):
            if mask[i,j]:
                i1, i2 = i*bh, min((i+1)*bh, H)
                j1, j2 = j*bw, min((j+1)*bw, W)
                result[i1:i2, j1:j2] = 0.5
    return result

def jitter(frame, px):
    result = frame.copy()
    H = frame.shape[0]
    shifts = (np.random.randn(H)*px).astype(int)
    for y in range(H):
        result[y] = np.roll(result[y], shifts[y], axis=0)
    return result

def bw_drop(frame, scale):
    H, W = frame.shape[:2]
    h2, w2 = int(H*scale), int(W*scale)
    low = ndimage.zoom(frame, (scale,scale,1), order=1)
    up = ndimage.zoom(low, (1/scale,1/scale,1), order=1)
    return np.clip(up[:H,:W], 0, 1)

print("Simulators ready (frame-size-aware)")


In [ ]:
# 对比：相同网络损伤对 VR 和 HD 的不同影响
np.random.seed(42)

# 从 VR 全景帧中裁切视场角区域（模拟头显中实际看到的画面）
fov_h, fov_w = 270, 480
cy, cx = EH//2, EW//2
hd_crop = vr_frame[cy-fov_h//2:cy+fov_h//2, cx-fov_w//2:cx+fov_w//2]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

impairments = [
    ('Original', lambda f: f),
    ('Loss 5%', lambda f: packet_loss(f, 0.05)),
    ('Jitter 5px', lambda f: jitter(f, 5)),
]

for j, (label, func) in enumerate(impairments):
    # 完整 VR 帧
    axes[0, j].imshow(np.clip(func(vr_frame), 0, 1))
    axes[0, j].set_title('Full VR: {}'.format(label), fontsize=11)
    axes[0, j].axis('off')
    # 视场角裁切（用户实际看到的部分）
    crop = np.clip(func(hd_crop), 0, 1)
    axes[1, j].imshow(crop)
    axes[1, j].set_title('FOV Crop: {}'.format(label), fontsize=11)
    axes[1, j].axis('off')
    # 中心物体放大
    ch, cw = crop.shape[0]//2, crop.shape[1]//2
    r = 60
    z = crop[max(0,ch-r):min(crop.shape[0],ch+r), max(0,cw-r):min(crop.shape[1],cw+r)]
    axes[2, j].imshow(z)
    axes[2, j].set_title('Center Zoom: {}'.format(label), fontsize=11)
    axes[2, j].axis('off')

plt.suptitle('VR vs HD: Same Impairment, Different Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("观察要点：")
print("1. VR 全景帧中，损伤在画面各处可见")
print("2. 在视场角裁切（用户实际观看区域）中，损伤更加集中和明显")
print("3. 中心物体放大后，即使 5% 的丢包也会破坏关键视觉信息")
print("4. VR 需要比 HD 高 4-8 倍的带宽才能达到同等感知质量")


In [ ]:
# VR 时延敏感性对比
def frame_drop_sim(frame, drop_ratio):
    return frame * (1.0 - drop_ratio * 0.3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(np.clip(vr_frame, 0, 1))
axes[0].set_title('Smooth VR (60 fps)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(np.clip(frame_drop_sim(vr_frame, 0.5), 0, 1))
axes[1].set_title('Frame Drops 50%\nMotion sickness risk!', fontsize=12)
axes[1].axis('off')

axes[2].imshow(np.clip(packet_loss(bw_drop(vr_frame, 0.5), 0.1), 0, 1))
axes[2].set_title('Severe: BW 50% + Loss 10%\nVR experience broken', fontsize=12)
axes[2].axis('off')

plt.suptitle('VR Quality Degradation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n为什么 VR 对网络损伤更敏感：")
print("1. 更高分辨率（4K-8K vs 1080p）-> 每帧数据量更大")
print("2. 超低时延要求（<20ms vs 流媒体 2-5s 缓冲）")
print("3. 帧丢失直接导致运动眩晕（前庭-视觉冲突）")
print("4. 等距矩形投影在两极浪费大量码率")
print("5. 注视点渲染假设传输完美无缺")


## 关键结论

- 5% 丢包在 HD 中几乎不可察觉，在 VR 中却明显可见
- 等距矩形投影在画面顶部/底部（极点）放大了压缩伪影
- VR 中的帧丢失直接导致运动眩晕——任何应用场景都无法接受
- VR 通常需要 HD 的 4-8 倍带宽才能达到同等感知质量

## 实际解决方案
- **注视点渲染**：只对用户注视区域传输高质量画面
- **视口依赖流式传输**：将 360 视频切分为瓦片，仅高质量传输可见瓦片
- **5G 边缘计算**：在网络边缘渲染 VR 内容，降低时延
- **Apple Vision Pro / Meta Quest**：使用专用无线芯片实现超低时延


In [ ]:
# 模拟更真实的 VR 全景场景
def create_realistic_vr():
    EH, EW = 480, 960
    frame = np.zeros((EH, EW, 3), dtype=np.float32)
    # 天空
    for y in range(EH//2):
        t = y/(EH//2)
        frame[y, :] = (0.2+0.5*t, 0.3+0.4*t, 0.5+0.4*t)
    # 远山（起伏轮廓）
    for x in range(EW):
        h = int(EH*0.35 + np.sin(x*0.008)*30 + np.sin(x*0.02)*20 + np.sin(x*0.05)*10)
        frame[h:EH//2+20, x] = (0.12, 0.28, 0.12)
    # 地面
    for y in range(EH//2+20, EH):
        v = 0.18 + 0.1*np.sin(y*0.04)
        frame[y, :] = (v, 0.22+v*0.4, v*0.4)
    # 建筑物散布在 360 空间中
    import random; random.seed(7)
    for _ in range(12):
        bx = random.randint(20, EW-100)
        bw = random.randint(40, 90)
        bh = random.randint(50, 130)
        by = int(EH*0.35 - bh + random.randint(-20, 30))
        c = random.uniform(0.3, 0.7)
        by = max(0, by)
        top = min(by, EH//2+20)
        bot = min(top+bh, EH//2+20)
        frame[top:bot, bx:bx+bw] = (c*0.6, c*0.5, c*0.4)
    # 中心区域标注（用户注视热点）
    cx, cy = EW//2, EH//2
    rr = 100
    for y in range(cy-rr, cy+rr):
        for x in range(cx-rr, cx+rr):
            if (x-cx)**2 + (y-cy)**2 < rr**2:
                if 0 <= y < EH and 0 <= x < EW:
                    frame[y, x] *= 1.2
    return np.clip(frame, 0, 1)

real_vr = create_realistic_vr()
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(real_vr)
ax.set_title('Realistic VR Equirectangular Scene')
ax.axis('off')
plt.show()
print("全景场景包含：天空、远山、地面、散布的建筑群、中心注视热点")


In [ ]:
# 在真实感 VR 场景上对比不同网络损伤
np.random.seed(99)
# 裁切视场角区域
cy, cx = EH//2, EW//2
fov = real_vr[cy-135:cy+135, cx-240:cx+240]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
scenarios = [
    ('Ideal VR', fov),
    ('Loss 5%', packet_loss(fov, 0.05)),
    ('Jitter 3px', jitter(fov, 3)),
    ('BW 50% + Loss 5%', packet_loss(bw_drop(fov, 0.5), 0.05)),
]
for j, (label, img) in enumerate(scenarios):
    axes[j].imshow(np.clip(img, 0, 1))
    axes[j].set_title(label, fontsize=12, fontweight='bold')
    axes[j].axis('off')
plt.suptitle('VR Viewport: Network Impairment Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nVR 体验对网络的严苛要求总结：")
print("1. 5% 丢包在 HD 中可接受，在 VR 中已明显破坏体验")
print("2. 3px 抖动对 480px 宽视场角影响显著（占比 0.6%）")
print("3. 带宽下降 50% 叠加丢包时，VR 画面已不可用")
print("4. 这解释了为什么 VR 需要 5G/WiFi 6E 级别的网络支撑")
